In [5]:
import os
import mujoco
import mujoco.viewer
import mediapy as media
import matplotlib.pyplot as plt
import time
import itertools
import numpy as np
from pathlib import Path

In [6]:
#Carga de los modelos de mujoco
#Parámetros de la piernas
#Implementar lo siguiente cuando aprenda a hacerlo
#from .config import R_HIP_LEN, R_KNEE_LEN, R_ANKLE_LEN,F_HIP_LEN, F_KNEE_LEN, F_ANKLE_LEN,SCALE

SCALE = 1
R_HIP_LEN = 0.166 * SCALE
R_KNEE_LEN = 0.231 * SCALE
R_ANKLE_LEN = 0.148 * SCALE
F_HIP_LEN =  0.121 * SCALE
F_KNEE_LEN = 0.111 * SCALE
#tener en cuenta o no tener en cuenta la escápla, eso añadiria ortro grado de libertad
#Le voy a sumar la longitud de los falanges en vez de añadir otro grado de liberad
F_ANKLE_LEN = (0.0802+0.0288+0.0188+0.0256) * SCALE 

FRAMERATE = 30

path = Path.cwd().parent / 'assets' / 'xml' / 'pata_trasera_bonilla_contact.xml'

with open(path, "r") as file:
    xml_temp = file.read()

base_values = {
    "thigh": (0.03, 1.68),
    "shank": (0.025, 0.795),
    "metatarsus": (0.02, 0.099)
}
# Compute scaled values
scaled_values_front = {}
for name, (size, mass) in base_values.items():
    scaled_values_front[f"{name}_size"] = size * SCALE
    scaled_values_front[f"{name}_mass"] = mass * SCALE

scaled_values_back = {}
for name, (size, mass) in base_values.items():
    scaled_values_back[f"{name}_size"] = size * SCALE
    scaled_values_back[f"{name}_mass"] = mass * SCALE

xml_front = xml_temp.format(
    thigh_len=F_HIP_LEN,
    shank_len=F_KNEE_LEN,
    metatarsus_len=F_ANKLE_LEN,
    plane_len=F_HIP_LEN+F_KNEE_LEN+F_ANKLE_LEN,
    **scaled_values_front
)

xml_back=xml_temp.format(
    thigh_len=R_HIP_LEN,
    shank_len=R_KNEE_LEN,
    metatarsus_len=R_ANKLE_LEN,
    plane_len=R_HIP_LEN+R_KNEE_LEN+R_ANKLE_LEN,
    **scaled_values_back
)

model_front_leg=mujoco.MjModel.from_xml_string(xml_front)
data_front = mujoco.MjData(model_front_leg)
model_back_leg=mujoco.MjModel.from_xml_string(xml_back)
data_back = mujoco.MjData(model_back_leg)

In [7]:
#np.random.seed(23)
data_front.qpos=[0,np.random.randn()/10,np.random.randn()/10,np.random.randn()/10]
mujoco.mj_forward(model_front_leg,data_front)
print(data_front.qpos)

[ 0.         -0.14220619 -0.09272013  0.12361382]


In [8]:
n_frames = 200
height = 240
width = 320
frames = []
renderer = mujoco.Renderer(model_front_leg, height, width)

# visualize contact frames and forces, make body transparent
options = mujoco.MjvOption()
mujoco.mjv_defaultOption(options)
options.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
options.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
options.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True

# tweak scales of contact visualization elements
model_front_leg.vis.scale.contactwidth = 0.1
model_front_leg.vis.scale.contactheight = 0.03
model_front_leg.vis.scale.forcewidth = 0.05
model_front_leg.vis.map.force = 0.3

mujoco.mj_forward(model_front_leg,data_front)
# simulate and render
for i in range(n_frames):
  while data_front.time < i/60.0: #1/4x real time
    mujoco.mj_step(model_front_leg, data_front)
    
  renderer.update_scene(data_front, "track", options)
  frame = renderer.render()
  frames.append(frame)

# show video
media.show_video(frames, fps=30)
renderer.close()
    

This browser does not support the video tag.


In [9]:
mujoco.mj_resetData

<function mujoco._functions.pybind11_detail_function_record_v1_msvc_md_mscver19.mj_resetData>